In [ ]:
CPP_SOURCE = r'''// Identify a molecule by finding the training spectra that look like it.
//
// The shape of the problem, measured before any of this was written: 400 test
// molecules, 1,213 spectra, all from one instrument; 2.54 million training
// spectra over 276 thousand molecules. Every test precursor mass exists in the
// training set, but within 10 ppm the median mass has 105 distinct molecules
// behind it and the worst has 558 - so the mass narrows the field to about a
// hundred and the spectrum has to do the rest. Only 38 of the 400 have few
// enough candidates that mass alone would fill the answer.
//
// That is a billion peak-list comparisons, which is why this is not Python.
//
// Build:
//   g++ -O3 -std=c++17 -pthread -o match match.cpp

#include <algorithm>
#include <atomic>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <fstream>
#include <string>
#include <thread>
#include <unordered_map>
#include <vector>

namespace {

struct Head {            // one per spectrum, as written by export_bin.py
  float precursor;
  int32_t mol;
  int32_t off;           // index into the peak arrays
  int32_t n;             // how many peaks
  int32_t tag;           // adduct * 16 + instrument, -1 where unknown
};

struct Store {
  std::vector<Head> head;
  std::vector<float> mz, inten;
  std::vector<std::string> name, smiles;

  const float* mzs(const Head& h) const { return mz.data() + h.off; }
  const float* its(const Head& h) const { return inten.data() + h.off; }
  int adduct(const Head& h) const { return h.tag < 0 ? -1 : h.tag / 16; }
};

std::vector<char> slurp(const std::string& path) {
  std::ifstream f(path, std::ios::binary | std::ios::ate);
  if (!f) { std::fprintf(stderr, "cannot read %s\n", path.c_str()); std::exit(1); }
  std::streamsize n = f.tellg();
  f.seekg(0);
  std::vector<char> buf((size_t(n)));
  f.read(buf.data(), n);
  return buf;
}

Store load(const std::string& dir, const std::string& stem) {
  Store s;
  std::vector<char> hb = slurp(dir + "/" + stem + "_head.bin");
  s.head.resize(hb.size() / sizeof(Head));
  std::memcpy(s.head.data(), hb.data(), s.head.size() * sizeof(Head));

  std::vector<char> pb = slurp(dir + "/" + stem + "_peaks.bin");
  // Each spectrum wrote its m/z block then its intensity block, so the pairs
  // are interleaved per spectrum rather than globally. Split them out once.
  size_t total = 0;
  for (const Head& h : s.head) total += size_t(h.n);
  s.mz.resize(total);
  s.inten.resize(total);
  const float* p = reinterpret_cast<const float*>(pb.data());
  size_t cur = 0;
  for (const Head& h : s.head) {
    std::memcpy(s.mz.data() + cur, p, size_t(h.n) * sizeof(float));
    p += h.n;
    std::memcpy(s.inten.data() + cur, p, size_t(h.n) * sizeof(float));
    p += h.n;
    cur += size_t(h.n);
  }

  std::ifstream f(dir + "/" + stem + "_mols.txt");
  std::string line;
  while (std::getline(f, line)) {
    if (!line.empty() && line.back() == '\r') line.pop_back();
    size_t t = line.find('\t');
    s.name.push_back(line.substr(0, t));
    s.smiles.push_back(t == std::string::npos ? "" : line.substr(t + 1));
  }
  std::printf("%s: %zu spectra, %zu molecules, %zu peaks\n",
              stem.c_str(), s.head.size(), s.name.size(), total);
  return s;
}

// Cosine over matched peaks, the standard way these libraries are searched.
//
// Peaks match when their m/z agree within a tolerance; both lists are sorted,
// so one walk of each is enough. Intensities are square-rooted first, which is
// what every spectral library search does - it stops one tall peak from
// deciding the whole score, and tall peaks are the least informative part of a
// fragmentation pattern.
float cosine(const float* amz, const float* ait, int an,
             const float* bmz, const float* bit, int bn, float tol) {
  double dot = 0.0, na = 0.0, nb = 0.0;
  for (int i = 0; i < an; ++i) na += double(ait[i]);
  for (int j = 0; j < bn; ++j) nb += double(bit[j]);
  if (na <= 0.0 || nb <= 0.0) return 0.0;

  int j = 0;
  for (int i = 0; i < an; ++i) {
    while (j < bn && bmz[j] < amz[i] - tol) ++j;
    int k = j;
    float best = 0.0f;
    while (k < bn && bmz[k] <= amz[i] + tol) {
      best = std::max(best, bit[k]);
      ++k;
    }
    if (best > 0.0f) dot += std::sqrt(double(ait[i])) * std::sqrt(double(best));
  }
  double norma = 0.0, normb = 0.0;
  for (int i = 0; i < an; ++i) norma += double(ait[i]);
  for (int j2 = 0; j2 < bn; ++j2) normb += double(bit[j2]);
  return float(dot / std::sqrt(norma * normb + 1e-12));
}

}  // namespace

int main(int argc, char** argv) {
  std::string dir = argc > 1 ? argv[1] : ".";
  float ppm = argc > 2 ? float(std::atof(argv[2])) : 10.0f;
  float tol = argc > 3 ? float(std::atof(argv[3])) : 0.01f;
  int topk = argc > 4 ? std::atoi(argv[4]) : 25;
  // Swap in the holdout by name, so the thing that is measured is the thing
  // that is submitted and not a second implementation of it.
  std::string lib_stem = argc > 5 ? argv[5] : "train";
  std::string qry_stem = argc > 6 ? argv[6] : "test";

  Store tr = load(dir, lib_stem);
  Store te = load(dir, qry_stem);

  // Training spectra sorted by precursor mass, so a candidate window is a
  // binary search rather than a scan of two and a half million rows.
  std::vector<int> order(tr.head.size());
  for (size_t i = 0; i < order.size(); ++i) order[i] = int(i);
  std::sort(order.begin(), order.end(),
            [&](int a, int b) { return tr.head[a].precursor < tr.head[b].precursor; });
  std::vector<float> keys(order.size());
  for (size_t i = 0; i < order.size(); ++i) keys[i] = tr.head[order[i]].precursor;

  // Group the test spectra by molecule: several spectra of one molecule are
  // evidence about the same answer and their scores add.
  std::unordered_map<int, std::vector<int> > by_mol;
  for (size_t i = 0; i < te.head.size(); ++i) by_mol[te.head[i].mol].push_back(int(i));

  std::vector<int> mol_ids;
  for (const auto& kv : by_mol) mol_ids.push_back(kv.first);
  std::sort(mol_ids.begin(), mol_ids.end());

  std::vector<std::string> answer(mol_ids.size());
  std::vector<int> hit_rank(mol_ids.size(), -1);
  unsigned nthread = std::max(1u, std::thread::hardware_concurrency());
  std::vector<std::thread> pool;
  std::atomic<size_t> next(0);

  for (unsigned t = 0; t < nthread; ++t) {
    pool.emplace_back([&]() {
      std::unordered_map<int, float> best;      // train molecule -> best score
      for (;;) {
        size_t idx = next++;
        if (idx >= mol_ids.size()) break;
        best.clear();

        // Widen the mass window until there are enough candidates to fill the
        // answer.
        //
        // A tight window is what made this work - at 0.2 ppm the median test
        // molecule competes against a handful of others instead of a hundred,
        // and MRR went from 0.084 to 0.44. But tight also means 354 of the 400
        // molecules came back with fewer than 25 candidates and one came back
        // with a single candidate, which throws away the twenty-four free
        // guesses the format allows. So start tight and open up only as far as
        // filling the list requires.
        // A candidate found in a tighter pass keeps the rank that pass gave
        // it: the tight window is the better evidence, and a molecule only
        // reachable at 50 ppm belongs below one reachable at 0.2.
        float ppm_used = ppm;
        for (int pass = 0; pass < 8 && int(best.size()) < topk; ++pass) {
          for (int si : by_mol[mol_ids[idx]]) {
            const Head& q = te.head[si];
            float window = q.precursor * ppm_used * 1e-6f;
            size_t lo = std::lower_bound(keys.begin(), keys.end(),
                                         q.precursor - window) - keys.begin();
            size_t hi = std::upper_bound(keys.begin(), keys.end(),
                                         q.precursor + window) - keys.begin();
            for (size_t p = lo; p < hi; ++p) {
              const Head& c = tr.head[order[p]];
              if (q.tag >= 0 && c.tag >= 0 && te.adduct(q) != tr.adduct(c)) continue;
              float s = cosine(te.mzs(q), te.its(q), q.n,
                               tr.mzs(c), tr.its(c), c.n, tol);
              // Later passes are worse evidence, so their scores sit below
              // everything the tight passes already placed.
              s -= float(pass);
              auto it = best.find(c.mol);
              if (it == best.end()) best.emplace(c.mol, s);
              else it->second = std::max(it->second, s);
            }
          }
          ppm_used *= 3.0f;
        }

        std::vector<std::pair<float, int> > rank;
        rank.reserve(best.size());
        for (const auto& kv : best) rank.emplace_back(kv.second, kv.first);
        std::sort(rank.begin(), rank.end(),
                  [](const std::pair<float, int>& a, const std::pair<float, int>& b) {
                    return a.first > b.first;
                  });

        std::string out;
        int placed = 0;
        for (size_t r = 0; r < rank.size() && placed < topk; ++r) {
          const std::string& smi = tr.smiles[rank[r].second];
          if (smi.empty()) continue;          // a molecule with no structure
          if (!out.empty()) out += ';';
          out += smi;
          ++placed;
          // On a holdout the query carries its own answer, so note where it
          // landed. The run then scores itself and no submission is spent
          // finding out whether a change helped.
          if (hit_rank[idx] < 0 && tr.name[rank[r].second] == te.name[mol_ids[idx]])
            hit_rank[idx] = placed;
        }
        answer[idx] = out;
      }
    });
  }
  for (std::thread& th : pool) th.join();

  std::string path = dir + "/submission.csv";
  std::ofstream out(path);
  out << "molecule_id,smiles\n";
  for (size_t i = 0; i < mol_ids.size(); ++i)
    out << te.name[mol_ids[i]] << ',' << answer[i] << '\n';
  std::printf("wrote %s for %zu molecules\n", path.c_str(), mol_ids.size());

  int found = 0;
  for (int r : hit_rank) found += (r > 0);
  if (found) {
    double mrr = 0.0;
    int t1 = 0, t5 = 0;
    for (int r : hit_rank) {
      if (r <= 0) continue;
      mrr += 1.0 / r;
      t1 += (r == 1);
      t5 += (r <= 5);
    }
    size_t n = mol_ids.size();
    std::printf("top1 %.1f%%  top5 %.1f%%  top%d %.1f%%  MRR %.3f  (n=%zu)\n",
                100.0 * t1 / n, 100.0 * t5 / n, topk, 100.0 * found / n,
                mrr / n, n);
  }
  return 0;
}
'''


In [ ]:
EXPORT_SOURCE = r'''r"""Turn the parquet into something C++ can mmap and scan.

Two and a half million spectra will be compared against four hundred, which is
a billion peak-list comparisons if done naively. Parquet and pandas are the
wrong shape for that: the work is a tight loop over float arrays, so the data
goes out as flat binary and the loop goes in C++.

Layout, all little-endian:
  head.bin   one record per spectrum, fixed width
  peaks.bin  the m/z and intensity pairs, concatenated, pointed into by head
  smiles.txt one line per molecule, indexed by head.mol
"""
from __future__ import annotations

import struct, sys
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq

ROOT = Path(sys.argv[1] if len(sys.argv) > 1 else r"C:\prog\casmi")

ADDUCTS = ["[M+H]+", "[M-H]-", "[M+Na]+", "[M+NH4]+", "[M+K]+", "[M+Cl]-",
           "[M+CH2O2-H]-", "[M+H-H2O]+", "[2M+H]+", "[2M+Na]+"]
ADDUCT_ID = {a: i for i, a in enumerate(ADDUCTS)}

INSTR = ["timsTOF", "Orbitrap", "QTOF", "LC-ESI-QTOF", "ESI-QFT"]
INSTR_ID = {a: i for i, a in enumerate(INSTR)}


def export(src: Path, out: str, with_labels: bool) -> None:
    cols = ["precursor_mz", "adduct", "instrument_type",
            "ms2_mzs", "ms2_normalized_intensities"]
    cols += ["inchikey14", "normalized_smiles"] if with_labels else ["molecule_id"]

    f = pq.ParquetFile(src)
    head = open(ROOT / f"{out}_head.bin", "wb")
    peaks = open(ROOT / f"{out}_peaks.bin", "wb")
    mols: dict[str, int] = {}
    names: list[str] = []
    smiles: list[str] = []
    off = 0
    n = 0

    for batch in f.iter_batches(batch_size=100000, columns=cols):
        d = batch.to_pandas()
        for r in d.itertuples(index=False):
            mz = np.asarray(r.ms2_mzs, dtype=np.float32)
            it = np.asarray(r.ms2_normalized_intensities, dtype=np.float32)
            k = min(len(mz), len(it))
            if k == 0:
                continue
            mz, it = mz[:k], it[:k]
            o = np.argsort(mz)            # the comparison walks both lists in order
            peaks.write(mz[o].tobytes())
            peaks.write(it[o].tobytes())

            if with_labels:
                name = r.inchikey14 if isinstance(r.inchikey14, str) else ""
                smi = r.normalized_smiles if isinstance(r.normalized_smiles, str) else ""
            else:
                name, smi = r.molecule_id, ""
            if name not in mols:
                mols[name] = len(names)
                names.append(name)
                smiles.append(smi)
            elif smi and not smiles[mols[name]]:
                smiles[mols[name]] = smi

            head.write(struct.pack(
                "<fiiii",
                float(r.precursor_mz) if r.precursor_mz == r.precursor_mz else 0.0,
                mols[name], off, k,
                ADDUCT_ID.get(r.adduct, -1) * 16 + INSTR_ID.get(r.instrument_type, -1)))
            off += k
            n += 1
        print(f"  {n} spectra", end="\r", flush=True)

    head.close()
    peaks.close()
    (ROOT / f"{out}_mols.txt").write_text(
        "\n".join(f"{a}\t{b}" for a, b in zip(names, smiles)), encoding="utf-8")
    print(f"\n{out}: {n} spectra, {len(names)} molecules, {off} peaks")


export(ROOT / "train.parquet", "train", True)
export(ROOT / "test.parquet", "test", False)
'''


In [ ]:
# Spectral library search, compiled here and run over the competition data.
#
# The task is a search, not a model: every one of the 400 test molecules has
# its precursor mass somewhere in the 2.5 million training spectra, and the
# work is picking the right one out of the others that share that mass. Which
# makes the whole thing a tight loop over 400 million peaks, so it is C++.
#
# What was measured on a held-out 400 molecules shaped like the test set:
#   10 ppm mass gate ..... MRR 0.084   (median molecule competes with 105)
#   0.05 ppm gate ........ MRR 0.474
#   plus staged widening . MRR 0.481,  top25 94%
# The gate is the whole story; the cosine barely changed.
import os, subprocess, sys, textwrap, time
from pathlib import Path

t0 = time.time()
WORK = Path("/kaggle/working")
(WORK / "match.cpp").write_text(CPP_SOURCE, encoding="utf-8")
(WORK / "export_bin.py").write_text(EXPORT_SOURCE, encoding="utf-8")

r = subprocess.run(["g++", "-O3", "-std=c++17", "-pthread", "-w",
                    "-o", str(WORK / "match"), str(WORK / "match.cpp")],
                   capture_output=True, text=True)
print("compile rc:", r.returncode, flush=True)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit(1)
print("compiled in %.0fs" % (time.time() - t0), flush=True)


In [ ]:
# Flatten the parquet into the binary the matcher scans.
#
# The competition directory is found rather than spelled out: the first attempt
# guessed the name from the URL slug and the run died having already compiled
# everything.
import glob, os
hits = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
print("inputs:", os.listdir("/kaggle/input"), flush=True)
if not hits:
    raise SystemExit("no train.parquet under /kaggle/input")
DATA = Path(hits[0]).parent
print("data:", DATA, flush=True)

for name in ("train.parquet", "test.parquet"):
    dst = WORK / name
    if not dst.exists():
        os.symlink(DATA / name, dst)

r = subprocess.run([sys.executable, str(WORK / "export_bin.py"), str(WORK)],
                   capture_output=True, text=True)
print(r.stdout[-1500:], flush=True)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit(1)
print("exported in %.0fs" % (time.time() - t0), flush=True)


In [ ]:
# 0.05 ppm to start, widening threefold per pass only until the answer list is
# full. A candidate found in a tighter pass outranks one that needed a looser
# pass, which is what the per-pass penalty inside the matcher is for.
r = subprocess.run([str(WORK / "match"), str(WORK), "0.05", "0.02", "25",
                    "train", "test"], capture_output=True, text=True)
print(r.stdout[-2000:], flush=True)
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise SystemExit(1)

import pandas as pd
sub = pd.read_csv(WORK / "submission.csv")
sub.to_csv(WORK / "submission.csv", index=False)
n = sub.smiles.fillna("").str.count(";") + 1
print("rows %d   full lists %d   fewest candidates %d   total %.0fs"
      % (len(sub), int((n == 25).sum()), int(n.min()), time.time() - t0))
sub.head(3)
